# ComfyUI 조건 텐서 → DiffSynth 하위 단계 비교

T4에서 공통 Python·torch·CUDA 환경으로 실행합니다. 저장소의 `tensor_replay/comfy_inputs/`를 사용하므로 입력 ZIP 업로드는 필요 없습니다.

- 최종 텍스트 조건: 공통 환경 ComfyUI ER-SDE 실행에서 추출한 가중치·패딩 적용 후 DiT 입력. 배치 순서는 negative, positive이며 positive는 원본 어댑터 출력·가중치로 재구성한 값과 정확히 일치합니다.
- 초기 노이즈·sigma 시간표·비교 이미지: 같은 환경의 ComfyUI Euler 실행. 두 실행의 모델·텍스트 입력·초기 노이즈 동일성을 확인했습니다.
- Qwen과 텍스트 어댑터는 실행하지 않고, DiffSynth의 원본 DiT·CFG·Euler·VAE를 실행합니다. 입력 출처와 SHA-256은 `manifest.json`에 있습니다.
- Diffusers에는 5번과 같은 잔차 FP32 패치를 적용합니다. NaN 디버그는 기본 OFF입니다. ComfyUI의 내부 계산 코드는 가져오지 않습니다.
- sigma 값은 동일하게 주입하지만 모델 입력 timestep의 dtype, CFG 연산, 샘플러 누적 정밀도, VAE는 각 구현을 따릅니다. DiffSynth는 초기 노이즈를 원본처럼 FP16으로 변환합니다. 남는 차이에 이 항목들이 포함됩니다.

실제 전체 모델의 T4 실행은 재확인이 필요합니다. 결과 ZIP에는 입력 출처, 주요 텐서, 로그, ComfyUI Euler 대비 수치가 포함됩니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys
from IPython.display import Image, display

REPOSITORY = 'https://github.com/HisameOgasahara/DiffFlowDiT_test.git'
REVISION = 'main'
PROJECT = Path('/content/DiffFlowDiT_original')
DATA = Path('/content/diffsynth_replay_data')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REVISION, REPOSITORY, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], check=True)
DATA.mkdir(parents=True, exist_ok=True)

# 설치·추론의 화면 출력과 로그 파일을 함께 기록합니다.
def launch(command, log_path):
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen([str(x) for x in command], cwd=PROJECT,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            if process.wait():
                raise RuntimeError(f'실행 실패: {log_path}')

## 환경 설치
기존 원본 실행용 독립 환경 설치 스크립트를 사용합니다.


In [ ]:
INSTALL_LOG = DATA / 'install.log'
launch([sys.executable, '-u', PROJECT / 'tools/setup_diffsynth_original.py'], INSTALL_LOG)
PYTHON = PROJECT / '.venv-diffsynth-original/bin/python'

## 고정 입력과 모델 준비
모델 준비는 기존 스크립트를 재사용합니다. 텍스트 가중치·토크나이저도 내려받지만 추론 시에는 로드하지 않습니다. 이 실험의 생성 설정은 입력 묶음과 일치해야 합니다.


In [ ]:
CONFIG = PROJECT / 'tensor_replay/comfy_inputs/generation.json'
RUNTIME = DATA / 'runtime.json'
runtime = json.loads((PROJECT / 'diffsynth/original_runtime.json').read_text(encoding='utf-8'))
runtime['nan_debug'] = False
RUNTIME.write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding='utf-8')
TRACE = 'selected'
MODELS = DATA / 'models'
launch([PYTHON, '-u', PROJECT / 'diffsynth/prepare_original.py',
        '--config', CONFIG, '--runtime', RUNTIME, '--models', MODELS], DATA / 'prepare.log')


## 텍스트 조건 이후 단계 실행
첫·둘째·마지막 스텝의 예측값과 latent, 최종 latent·이미지를 저장합니다.


In [ ]:
RUN_NAME = 'diffsynth_comfy_replay_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = DATA / 'runs' / RUN_NAME
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RUN_LOG = DATA / (RUN_NAME + '.log')
try:
    launch([PYTHON, '-u', PROJECT / 'tensor_replay/run_diffsynth.py', '--config', CONFIG,
            '--runtime', RUNTIME, '--models', MODELS, '--output', OUTPUT,
            '--mode', 'native', '--trace', TRACE], RUN_LOG)
finally:
    if OUTPUT.exists():
        shutil.copy2(RUN_LOG, OUTPUT / 'run.log')


In [ ]:
print('ComfyUI Euler 기준 이미지')
display(Image(filename=str(PROJECT / 'tensor_replay/comfy_inputs/reference.png')))
if (OUTPUT / 'image.png').exists():
    print('현재 프레임워크 결과')
    display(Image(filename=str(OUTPUT / 'image.png')))
    print((OUTPUT / 'comparison.json').read_text(encoding='utf-8'))


## 결과 ZIP 다운로드
실행이 실패해도 이 셀을 따로 실행하면 남은 로그를 받을 수 있습니다.


In [ ]:
DOWNLOAD = True
ARCHIVE = shutil.make_archive(str(DATA / RUN_NAME), 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name)
print(ARCHIVE)
if DOWNLOAD:
    from google.colab import files
    files.download(ARCHIVE)
